# Exercise 7: motif A or motif B, but not both

Create a new artificial DNA dataset and compare a **classic perceptron** with a **multilayer perceptron (MLP)**. A sequence is positive when it contains exactly one of two motifs. Both motifs, or neither motif, means negative. This is **exclusive OR (XOR)**.

| Motif A | Motif B | Label |
| --- | --- | --- |
| Absent | Absent | 0 |
| Present | Absent | 1 |
| Absent | Present | 1 |
| Present | Present | 0 |

This is a synthetic regulatory task, not a claim about how these sequences regulate real genes. Biological XOR has been engineered, for example in [Tamsir, Tabor and Voigt (2011)](https://doi.org/10.1038/nature09565).

Run cells from top to bottom. No CSV upload or earlier notebook is needed. The dataset is saved before any models are trained. **Run all includes the final test evaluation.**


## 1. Import libraries and choose settings

Use two fixed 10-base motifs with the same nucleotide composition. Unlike Exercise 1, we use exact sequences rather than sampling from a PFM. There will be 5,000 matched backgrounds and four variants per background: **20,000 sequences**, each 100 nt long.

We keep k=6 fixed. The MLP has one hidden layer with 16 ReLU units and a sigmoid output for binary classification. All libraries are already used by the earlier exercises.


In [ ]:
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, log_loss,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay,
)

seed = 42
n_backgrounds = 5_000
sequence_length = 100
motif_a = "ACGTCAGTCA"
motif_b = "GACTACCTGA"
k = 6
max_epochs = 60
ofile = "exercise_7_output/dataset.csv"

motif_length = len(motif_a)
assert motif_a != motif_b
assert len(motif_b) == motif_length
assert sorted(motif_a) == sorted(motif_b)
assert set(motif_a + motif_b) <= set("ACGT")
assert 1 <= k <= motif_length
assert sequence_length >= 2 * motif_length + k - 1


## 2. Make four variants of each background

Sample background bases independently with A=C=G=T=25%. Choose two random insertion sites. They must leave at least k-1 background bases between the motifs, so **no k-mer can touch both insertion sites**. Motifs replace bases and use the supplied orientation only.

For each background, make all four cases at the same two sites. Reject the entire set if an unintended motif occurs, including across an insertion boundary. Count overlapping exact matches when checking. Each accepted motif occurs exactly once when present and zero times when absent.

The matched design prevents background composition from becoming a shortcut. `has_a`, `has_b` and `background_id` are metadata; the k-mer models will receive only `sequence`.


In [ ]:
rng = np.random.default_rng(seed)
rows = []
cases = [(0, 0), (1, 0), (0, 1), (1, 1)]

while len(rows) < 4 * n_backgrounds:
    start_a, start_b = rng.choice(
        sequence_length - motif_length + 1, size=2, replace=False,
    )
    if abs(start_a - start_b) < motif_length + k - 1:
        continue
    background = rng.choice(list("ACGT"), size=sequence_length)
    background_id = len(rows) // 4
    variants = []

    for has_a, has_b in cases:
        bases = background.copy()
        if has_a:
            bases[start_a:start_a + motif_length] = list(motif_a)
        if has_b:
            bases[start_b:start_b + motif_length] = list(motif_b)
        sequence = "".join(bases)
        count_a = sum(
            sequence[p:p + motif_length] == motif_a
            for p in range(sequence_length - motif_length + 1)
        )
        count_b = sum(
            sequence[p:p + motif_length] == motif_b
            for p in range(sequence_length - motif_length + 1)
        )
        if count_a != has_a or count_b != has_b:
            break
        variants.append({
            "sequence": sequence, "label": int(has_a != has_b),
            "has_a": has_a, "has_b": has_b,
            "background_id": background_id,
        })

    if len(variants) == 4:
        rows.extend(variants)

dataset = pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)
print("Sequences:", len(dataset))
print(pd.crosstab([dataset["has_a"], dataset["has_b"]], dataset["label"]))
dataset.head()


## 3. Check and save the new dataset

There are 5,000 sequences per case and 10,000 per label. Keep `background_id` in the CSV so related sequences can stay together when splitting. In Colab, download the CSV from the Files sidebar after this cell.


In [ ]:
assert len(dataset) == 4 * n_backgrounds
assert dataset["sequence"].str.fullmatch(f"[ACGT]{{{sequence_length}}}").all()
assert dataset["sequence"].is_unique
assert dataset.groupby(["has_a", "has_b"]).size().eq(n_backgrounds).all()
assert dataset["label"].eq((dataset["has_a"] != dataset["has_b"]).astype(int)).all()

output_path = Path(ofile)
output_path.parent.mkdir(parents=True, exist_ok=True)
dataset.to_csv(output_path, index=False)
print("Saved:", output_path)


## 4. See why XOR needs nonlinearity

First use just the two motif-presence indicators. The classic perceptron thresholds a weighted sum; logistic regression applies a sigmoid to a weighted sum and learns with log loss. Both have linear decision boundaries in these inputs. **No line can classify all four XOR corners correctly.** A line can classify three corners, so linear does not necessarily mean 50% accuracy.

For a concrete two-unit hidden layer, set s=A+B, h1=ReLU(s), h2=ReLU(s-1), and output=h1-2*h2. The output equals XOR on the four binary inputs. These weights are written by hand to show what a hidden layer can represent; later the MLP will learn its weights.


In [ ]:
truth_table = pd.DataFrame(cases, columns=["has_a", "has_b"])
truth_table["label"] = (truth_table["has_a"] != truth_table["has_b"]).astype(int)
presence_perceptron = Perceptron(max_iter=1_000, tol=1e-3, random_state=seed)
presence_perceptron.fit(truth_table[["has_a", "has_b"]], truth_table["label"])
truth_table["perceptron"] = presence_perceptron.predict(truth_table[["has_a", "has_b"]])

s = truth_table["has_a"] + truth_table["has_b"]
h1 = np.maximum(0, s)
h2 = np.maximum(0, s - 1)
truth_table["two_relu_units"] = h1 - 2 * h2
print(truth_table.to_string(index=False))

fig, ax = plt.subplots(figsize=(5, 5))
for label, color in [(0, "tab:blue"), (1, "tab:orange")]:
    points = truth_table[truth_table["label"] == label]
    ax.scatter(points["has_a"], points["has_b"], s=180, color=color, label=f"Label {label}")
ax.set(xlabel="Motif A present", ylabel="Motif B present",
       xticks=[0, 1], yticks=[0, 1], xlim=(-0.3, 1.3), ylim=(-0.3, 1.3),
       title="Can one straight line separate the two labels?")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Split backgrounds, then select their sequences

Reserve 10% of background IDs for test and 20% for validation; use the remaining 70% for training. **All four variants of a background stay in the same split.** A random split of individual rows would put nearly identical background sequences into different splits.

Each split is balanced across all four cases. Use validation data to choose the MLP epoch. The test sequences stay unused until the final cell.


In [ ]:
background_ids = np.random.default_rng(seed).permutation(n_backgrounds)
n_test = int(0.10 * n_backgrounds)
n_validation = int(0.20 * n_backgrounds)
assert n_test > 0 and n_validation > 0
test_ids = background_ids[:n_test]
validation_ids = background_ids[n_test:n_test + n_validation]
train_ids = background_ids[n_test + n_validation:]

train_data = dataset[dataset["background_id"].isin(train_ids)].copy()
validation_data = dataset[dataset["background_id"].isin(validation_ids)].copy()
test_data = dataset[dataset["background_id"].isin(test_ids)].copy()
y_train = train_data["label"]
y_validation = validation_data["label"]

print("Training:", len(train_data))
print("Validation:", len(validation_data))
print("Test (reserved):", len(test_data))
print(train_data.groupby(["has_a", "has_b"]).size())


## 6. Give both models the same bag of 6-mers

Count overlapping k-mers, fitting the vocabulary on training sequences only. Each 100-nt sequence has 95 windows at k=6, with at most 4,096 possible features. Keep the counts sparse.

Neither model receives motif identities, presence flags, positions, or background IDs. They must learn from the sequence counts. Because no k-mer touches both sites, each matched set satisfies **x_neither + x_both = x_A_only + x_B_only**. Therefore a linear score cannot classify all four variants correctly, even in the full k-mer feature space.

The limitation here is combining two signals nonlinearly. In a later spacing task, the challenge will instead be information about arrangement that short-k-mer counts discard.


In [ ]:
vectorizer = CountVectorizer(
    analyzer="char", ngram_range=(k, k), lowercase=False, dtype=np.float32,
)
x_train = vectorizer.fit_transform(train_data["sequence"])
x_validation = vectorizer.transform(validation_data["sequence"])
print("Training count matrix:", x_train.shape)
print("Validation count matrix:", x_validation.shape)


## 7. Fit the classic perceptron

This is scikit-learn's classic perceptron, not logistic regression. It learns a linear score and uses zero as its decision threshold. It does not produce calibrated probabilities or optimize binary cross-entropy. Its score can still be used for ROC-AUC and average precision.


In [ ]:
perceptron_model = Perceptron(max_iter=1_000, tol=1e-3, random_state=seed)
perceptron_model.fit(x_train, y_train)
print("Perceptron validation accuracy:",
      round(accuracy_score(y_validation, perceptron_model.predict(x_validation)), 4))


## 8. Train an MLP one epoch at a time

The network computes **h=ReLU(W1 x+b1)** followed by **p=sigmoid(W2 h+b2)**. The hidden layer can combine motif-related features nonlinearly. It learns all weights from training labels, without labels for individual hidden units.

`partial_fit` performs one epoch: it processes mini-batches, calculates predictions and binary cross-entropy, computes gradients by backpropagation, and updates weights using Adam. Backpropagation computes gradients; Adam uses them to update parameters.

Keep 16 hidden units, batch size 128, learning rate 0.001, and L2 penalty alpha=0.0001 fixed. Record training and validation log loss after each epoch. Save a copy whenever validation loss improves; ties keep the earlier epoch. We select only the epoch and do not refit on validation data.


In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(16,), activation="relu", solver="adam",
    batch_size=128, learning_rate_init=0.001, alpha=0.0001,
    random_state=seed,
)
training_losses = []
validation_losses = []
best_validation_loss = np.inf

for epoch in range(1, max_epochs + 1):
    mlp.partial_fit(x_train, y_train, classes=[0, 1])
    training_loss = log_loss(y_train, mlp.predict_proba(x_train))
    validation_loss = log_loss(y_validation, mlp.predict_proba(x_validation))
    training_losses.append(training_loss)
    validation_losses.append(validation_loss)
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_epoch = epoch
        mlp_model = deepcopy(mlp)
    if epoch == 1 or epoch % 10 == 0:
        print(f"Epoch {epoch}: training={training_loss:.4f}, validation={validation_loss:.4f}")

print("Selected epoch:", best_epoch)
print("MLP parameters:", sum(w.size for w in mlp_model.coefs_)
      + sum(b.size for b in mlp_model.intercepts_))
epochs = range(1, max_epochs + 1)
plt.figure(figsize=(9, 5))
plt.plot(epochs, training_losses, label="Training")
plt.plot(epochs, validation_losses, label="Validation")
plt.axvline(best_epoch, color="black", linestyle="--", label="Selected epoch")
plt.xlabel("Epoch")
plt.ylabel("Binary log loss (without L2 penalty)")
plt.legend()
plt.tight_layout()
plt.show()


## 9. Compare validation performance, including each XOR case

Use the perceptron's decision score and the MLP's positive-class probability for ranking metrics. Use each model's `predict` for class calls: threshold zero for the perceptron, probability 0.5 for the MLP. We do not convert perceptron scores into probabilities or report log loss for them.

Inspect accuracy separately for neither, A only, B only, and both. Does an overall score hide one poorly predicted case? The matched design gives a linear classifier an accuracy ceiling of 75%; it does not guarantee that training will reach that ceiling.


In [ ]:
models = {"Perceptron": perceptron_model, "MLP": mlp_model}
validation_scores = {
    "Perceptron": perceptron_model.decision_function(x_validation),
    "MLP": mlp_model.predict_proba(x_validation)[:, 1],
}
rows = []
validation_case_results = validation_data[["has_a", "has_b"]].copy()
fig, axes = plt.subplots(2, 3, figsize=(17, 10))

for row, (name, model) in enumerate(models.items()):
    predictions = model.predict(x_validation)
    scores = validation_scores[name]
    validation_case_results[name] = (predictions == y_validation.to_numpy()).astype(int)
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_validation, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_validation, predictions),
        "precision": precision_score(y_validation, predictions, zero_division=0),
        "recall": recall_score(y_validation, predictions, zero_division=0),
        "f1": f1_score(y_validation, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_validation, scores),
        "average_precision": average_precision_score(y_validation, scores),
    })
    RocCurveDisplay.from_predictions(y_validation, scores, ax=axes[row, 0], name=name)
    axes[row, 0].plot([0, 1], [0, 1], "k--")
    PrecisionRecallDisplay.from_predictions(y_validation, scores, ax=axes[row, 1], name=name)
    axes[row, 1].axhline(y_validation.mean(), color="black", linestyle="--")
    ConfusionMatrixDisplay.from_predictions(
        y_validation, predictions, labels=[0, 1], display_labels=["Negative", "Positive"],
        ax=axes[row, 2], colorbar=False, cmap="Blues",
    )
    axes[row, 0].set_title(f"{name}: validation ROC")
    axes[row, 1].set_title(f"{name}: validation precision-recall")
    axes[row, 2].set_title(f"{name}: validation confusion matrix")

validation_results = pd.DataFrame(rows).set_index("model")
print(validation_results.round(4).to_string())
print("\nValidation accuracy by motif presence:")
print(validation_case_results.groupby(["has_a", "has_b"]).mean().round(4))
print("Selected MLP validation log loss:", round(best_validation_loss, 4))
plt.tight_layout()
plt.show()


## 10. STOP before the final test evaluation

Finish discussing validation results and selecting the MLP epoch first. Then evaluate both fitted models once on the reserved test backgrounds. Do not choose k, thresholds, hidden units, or epochs using test results. **Run all includes this cell.**


In [ ]:
x_test = vectorizer.transform(test_data["sequence"])
y_test = test_data["label"]
test_scores = {
    "Perceptron": perceptron_model.decision_function(x_test),
    "MLP": mlp_model.predict_proba(x_test)[:, 1],
}
rows = []
test_case_results = test_data[["has_a", "has_b"]].copy()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for column, (name, model) in enumerate(models.items()):
    predictions = model.predict(x_test)
    test_case_results[name] = (predictions == y_test.to_numpy()).astype(int)
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, test_scores[name]),
        "average_precision": average_precision_score(y_test, test_scores[name]),
    })
    ConfusionMatrixDisplay.from_predictions(
        y_test, predictions, labels=[0, 1], display_labels=["Negative", "Positive"],
        ax=axes[column], colorbar=False, cmap="Blues",
    )
    axes[column].set_title(f"{name}: test confusion matrix")

test_results = pd.DataFrame(rows).set_index("model")
print(test_results.round(4).to_string())
print("\nTest accuracy by motif presence:")
print(test_case_results.groupby(["has_a", "has_b"]).mean().round(4))
print("MLP test log loss:", round(log_loss(y_test, test_scores["MLP"]), 4))
plt.tight_layout()
plt.show()


## Discussion

- Why does detecting motif A alone fail when both motifs are present?
- What can a ReLU hidden layer represent that a single linear score cannot?
- Would adding more layers help if every hidden activation were linear?
- If we made k large enough to span both motifs, would the linear limitation still follow?
- How would a task based on motif spacing differ from this XOR task?

For a possible next exercise, keep both motifs present and vary their spacing. A TATA-like element and a transcription-start-site element provide a biological analogy: transcription initiation depends on promoter arrangement. A start codon concerns translation instead. Short-k-mer counts discard the relationship when no window spans both elements; a spacing dataset must control background and boundary clues carefully.
